In [ ]:
from matplotlib.lines import Line2D
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import random

# Muat data node dari file Excel Purwokerto_Nodes.xlsx
nodes_df = pd.read_excel("Purwokerto_Nodes.xlsx")
print(f"Loaded {len(nodes_df)} nodes from Purwokerto_Nodes.xlsx")

# Mengambil data peta dengan OSMnx dari titik pusat kota dengan radius 2000 meter.
print("Loading Purwokerto map data...")
city_center = (-7.4279011, 109.2408501)
G = ox.graph_from_point(city_center, dist=2000, network_type="drive")
# Ambil komponen terbesar agar data tersambung secara optimal
largest_component_nodes = max(nx.strongly_connected_components(G), key=len)
G = G.subgraph(largest_component_nodes).copy()

# OSMID untuk titik asal (Kedungwringin, Kabupaten Banyumas, Jawa Tengah)
origin_osmid = 8597363649     
# OSMID untuk titik tujuan (Purwokerto Lor, Kabupaten Banyumas, Jawa Tengah)
destination_osmid = 264506796

# Ambil koordinat dari Excel berdasarkan OSMID
origin_coords = nodes_df[nodes_df['osmid'] == origin_osmid][['y', 'x']].values[0]
dest_coords = nodes_df[nodes_df['osmid'] == destination_osmid][['y', 'x']].values[0]

# Cari node terdekat pada graph dari hasil koordinat diatas.
origin_node = ox.distance.nearest_nodes(G, origin_coords[1], origin_coords[0])
destination_node = ox.distance.nearest_nodes(G, dest_coords[1], dest_coords[0])
print(f"Origin node: {origin_node}, Destination node: {destination_node}")

# Salin graph asli menjadi graph yang akan dimodifikasi.
G_challenging = G.copy()

# --- Simulasi Blokade Node ---
# Buat daftar semua node, kecuali node asal dan tujuan.
all_nodes = list(G.nodes())
eligible_nodes = [n for n in all_nodes if n not in (origin_node, destination_node)]
# Hitung jumlah node yang ingin diblokade berdasarkan 30% dari node yang memenuhi syarat.
num_to_block = int(0.30 * len(eligible_nodes))
# Pilih secara acak sebanyak minimal antara num_to_block dan 50 node
blocked_nodes = random.sample(eligible_nodes, min(num_to_block, 50))
# Hapus node-node yang dipilih dari graph tantangan
for node in blocked_nodes:
    if G_challenging.has_node(node):
        G_challenging.remove_node(node)

# --- Simulasi Lalu Lintas Tinggi pada Edge ---
# Ambil daftar semua edge yang ada pada graph, termasuk data dan key.
all_edges = list(G.edges(data=True, keys=True))
# Buat list untuk menyimpan edge yang terpengaruh trafik tinggi.
high_traffic_edges = []
# Pilih secara acak 20% dari total edge sebagai edge dengan trafik tinggi.
sampled_edges = random.sample(all_edges, int(0.20 * len(all_edges)))
for u, v, k, data in sampled_edges:
    if G_challenging.has_edge(u, v, k):
        # Perpanjang panjang edge untuk menciptakan simulasi kemacetan (berarti waktu tempuh meningkat)
        G_challenging[u][v][k]['length'] *= 3
        high_traffic_edges.append((u, v, k))

route_dijkstra = None
try:
    # Cari rute terpendek berdasarkan atribut 'length'
    route_dijkstra = nx.shortest_path(G_challenging, origin_node, destination_node,
                                      weight='length', method='dijkstra')
    # Hitung total panjang rute yang ditemukan
    dijkstra_length = sum(G_challenging[u][v][0]['length'] for u, v in zip(route_dijkstra[:-1],
                                                                           route_dijkstra[1:]))
    print(f"Dijkstra (macet + blokade): {dijkstra_length:.2f} meters")
except nx.NetworkXNoPath:
    print("No path found using Dijkstra in the challenging graph.")

fig, ax = plt.subplots(figsize=(12, 10))

# Plot peta dasar dari graph asli dengan warna abu-abu.
ox.plot_graph(G, ax=ax, node_size=0, edge_color='gray', edge_linewidth=0.5,
              edge_alpha=0.5, show=False, close=False)

# Konversi graph ke GeoDataFrames untuk node dan edge.
nodes_gdf = ox.graph_to_gdfs(G, nodes=True, edges=False)
edges_gdf = ox.graph_to_gdfs(G, nodes=False)
# Reset index agar kolom 'u', 'v', dan 'key' tersedia sebagai kolom biasa.
edges_gdf = edges_gdf.reset_index()

# Plot node yang diblokade sebagai titik merah.
for node in blocked_nodes:
    if node in nodes_gdf.index:
        ax.scatter(nodes_gdf.loc[node].geometry.x, nodes_gdf.loc[node].geometry.y,
                   c='red', s=50, zorder=5)

# Plot edge dengan trafik tinggi sebagai garis berwarna orange.
for u, v, k in high_traffic_edges:
    # Cari baris edge pada GeoDataFrame berdasarkan kolom 'u', 'v', dan 'key'
    edge_row = edges_gdf[(edges_gdf['u'] == u) & (edges_gdf['v'] == v) & (edges_gdf['key'] == k)]
    if not edge_row.empty:
        # Jika geometry ada, gunakan koordinat geometry untuk menggambar garis
        if "geometry" in edge_row.columns and edge_row.iloc[0].geometry is not None:
            xs, ys = edge_row.iloc[0].geometry.xy
            ax.plot(xs, ys, color='orange', linewidth=3, zorder=4)
        else:
            # Jika geometry tidak ada, gambarkan garis lurus antara dua node
            if u in nodes_gdf.index and v in nodes_gdf.index:
                x_values = [nodes_gdf.loc[u].geometry.x, nodes_gdf.loc[v].geometry.x]
                y_values = [nodes_gdf.loc[u].geometry.y, nodes_gdf.loc[v].geometry.y]
                ax.plot(x_values, y_values, color='orange', linewidth=3, zorder=4)

# Tandai node asal dan tujuan
origin_point = nodes_gdf.loc[origin_node].geometry
dest_point = nodes_gdf.loc[destination_node].geometry
ax.scatter(origin_point.x, origin_point.y, c='darkgreen', s=50, marker='o', zorder=6,
           edgecolors='k', label="Origin")
ax.scatter(dest_point.x, dest_point.y, c='green', s=200, marker='*', zorder=6,
           edgecolors='k', label="Destination")

# Plot rute terpendek (jika ditemukan) dengan garis biru.
if route_dijkstra is not None:
    ox.plot_graph_route(G, route_dijkstra, ax=ax, route_color='blue',
                        route_linewidth=4, route_alpha=0.7, orig_dest_size=0)

# Sesuaikan batas tampilan berdasarkan total bounds node.
xmin, ymin, xmax, ymax = nodes_gdf.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

legend_elements = []
if route_dijkstra is not None:
    legend_elements.append(Line2D([0], [0], color='blue', linewidth=4, label='Dijkstra'))
legend_elements.extend([
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red',
           markersize=10, label='Blocked Node'),
    Line2D([0], [0], color='orange', linewidth=3, label='High Traffic Road'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='darkgreen',
           markersize=5, label='Origin'),
    Line2D([0], [0], marker='*', color='w', markerfacecolor='green',
           markersize=20, label='Destination')
])
ax.legend(handles=legend_elements)
plt.show()